# einops-repeat composite — cx2: outer-product two 1-D grids via repeat + new-axis broadcast

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-repeat`, `einops-repeat-broadcast`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-repeat"
DD_ATOM_IDS = ["einops-repeat", "einops-repeat-broadcast"]
DD_SUBTOPICS = ["Einops: Repeat", "Einops: Repeat-as-broadcast"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`einops-repeat` adds a new named axis to the output pattern and supplies its size via kwarg: `repeat(x, 'm -> m n', n=N)`. `einops-repeat-broadcast` is the *insight* that the inserted axis is a **stride-0 view** — no data is copied, einops just expands strides.

**The composition for outer products.** To form an outer product `O[i, j] = u[i] * v[j]` you need both vectors expanded to a common `(M, N)` shape, then multiplied elementwise:
1. `u: (M,)` becomes `(M, N)` via `repeat(u, 'm -> m n', n=N)`.
2. `v: (N,)` becomes `(M, N)` via `repeat(v, 'n -> m n', m=M)`.
3. Both expansions are stride-0 along the inserted axis, so the multiply is the only real work.

### Composite Exercise — outer-product two 1-D grids via repeat + new-axis broadcast

**Atoms exercised together**: `einops-repeat`, `einops-repeat-broadcast`

Build `cx2_outer_product(u, v)` that returns the outer product `O[i, j] = u[i] * v[j]` for 1-D tensors `u: (M,)` and `v: (N,)` using TWO `repeat` calls + one elementwise multiply.

Constraints:
- Must use `einops.repeat` for BOTH expansions (no `unsqueeze`, no `.expand`, no `torch.outer`).
- The returned tensor must have shape `(M, N)` and values matching `torch.outer(u, v)`.
- The two intermediate expansions must be stride-0 views (no copy) along the newly-inserted axis.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx2_outer_product(u, v):
    """Outer product via two repeat-broadcasts + elementwise multiply."""
    raise NotImplementedError()

def _test_cx2():
    # --- (a) basic correctness against torch.outer ---
    u = t.tensor([1.0, 2.0, 3.0])
    v = t.tensor([10.0, 20.0, 30.0, 40.0])
    out = cx2_outer_product(u, v)
    assert out.shape == (3, 4), f'expected (3,4), got {tuple(out.shape)}'
    assert t.allclose(out, t.outer(u, v)), f'outer-product mismatch: {out}'

    # --- (b) random sanity ---
    u2 = t.randn(7)
    v2 = t.randn(5)
    out2 = cx2_outer_product(u2, v2)
    assert out2.shape == (7, 5)
    assert t.allclose(out2, t.outer(u2, v2), atol=1e-5)

    # --- (c) value spot-check ---
    u3 = t.tensor([2.0, 5.0])
    v3 = t.tensor([3.0, 7.0])
    out3 = cx2_outer_product(u3, v3)
    expected = t.tensor([[6.0, 14.0], [15.0, 35.0]])
    assert t.allclose(out3, expected), f'spot-check failed: {out3}'
    _dd_passed.add('cx2')

_test_cx2()

<details><summary>Show solution — cx2</summary>

```python
def cx2_outer_product(u, v):
    M, N = u.shape[0], v.shape[0]
    U = repeat(u, 'm -> m n', n=N)   # (M, N), stride 0 along axis 1
    V = repeat(v, 'n -> m n', m=M)   # (M, N), stride 0 along axis 0
    return U * V
```

Both atoms compose in one expression. `repeat(...)` with a new output axis is the repeat atom; the stride-0 nature of the inserted axis is the repeat-broadcast atom. The multiply is allowed to allocate (it must materialize the M*N output), but the two `repeat`s themselves do NOT copy — they emit broadcast views.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx2',
        'subtopics': ["Einops: Repeat", "Einops: Repeat-as-broadcast"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()